In [10]:
#加载千问模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import torch

import os

device = "cuda"
model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen2___5-0___5B"

#根据当前环境选择合适的计算数据类型，如果支持bfloat16则使用bfloat16，否则使用float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
#配置量化参数，使用4-bit量化，选择nf4量化类型，启用双重量化，并设置计算数据类型为之前选择的compute_dtype。
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=compute_dtype,
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
  model_path,
  quantization_config=bnb_config, #使用之前配置的量化参数加载模型，这将使模型在加载时应用4-bit量化，从而减少显存占用并加速推理。
  device_map="auto",
  trust_remote_code=True,
)
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

model.config.use_cache = False #在训练过程中禁用缓存机制
model.gradient_checkpointing_enable()  #启用梯度检查点功能，这可以在训练过程中节省显存，但会增加计算时间。

#准备模型进行k-bit训练，这个函数会对模型进行必要的调整，以适应k-bit量化训练的要求，例如冻结某些层的权重，或者调整优化器的参数等。
model = prepare_model_for_kbit_training(model)

#配置QLora微调参数
lora_config = LoraConfig(
  r=16,#lora矩阵的秩，表示在低秩分解中使用的秩值。
  lora_alpha=32, #lora矩阵的缩放因子，通常设置为一个较大的值，以增强lora矩阵的影响力。
  lora_dropout=0.05, #在训练过程中应用于lora矩阵的dropout概率，帮助防止过拟合。
  bias="none", #指定是否微调模型中的偏置项，"none"表示不微调任何偏置项，"all"表示微调所有偏置项，"lora_only"表示只微调与lora相关的偏置项。
  task_type=TaskType.CAUSAL_LM, #指定微调任务的类型，这里设置为因果语言建模（Causal Language Modeling），表示模型将被微调用于生成文本。
  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], #指定要微调的模型模块列表，这里选择了与注意力机制相关的四个模块
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters() #打印模型中可训练参数的数量和占比，帮助我们了解微调过程中实际更新的参数规模。


#模型权重设置
print(model.generation_config)

model.generation_config.do_sample = False #是否使用采样方式生成文本，默认为False，即使用贪心算法生成文本。设置为True后，可以使用top_k和top_p参数来控制生成文本的多样性。
model.generation_config.eos_token_id = [151645, 151643] #模型生成文本时的结束标志，默认为None。可以设置为一个整数或一个整数列表，表示生成文本时的结束标志的token_id。当生成文本中出现这些token_id时，生成过程将停止。
model.generation_config.pad_token_id = 151643 #模型生成文本时的填充标志，默认为None。可以设置为一个整数，表示生成文本时的填充标志的token_id。当输入文本长度不足时，可以使用这个token_id进行填充。
model.generation_config.temperature = 0.7 #控制生成文本的随机程度，默认为1.0。较高的温度值会增加生成文本的多样性，而较低的温度值会使生成文本更集中和确定。通常，温度值在0.7到1.0之间被认为是一个合理的选择。
model.generation_config.top_p = 0.8 #控制生成文本的多样性，默认为1.0。top_p采样是一种基于概率的采样方法，它会根据生成文本的概率分布选择前p%概率的token进行采样。
model.generation_config.top_k = 20 #控制生成文本的多样性，默认为0。top_k采样是一种基于概率的采样方法，它会根据生成文本的概率分布选择前k个最可能的token进行采样。当top_k为0时，表示不使用top_k采样。
model.generation_config.repetition_penalty = 1.05 #控制生成文本的重复程度，默认为1.0。较高的重复惩罚值会增加生成文本的多样性，而较低的重复惩罚值会使生成文本更集中和确定。通常，重复惩罚值在1.0到2.0之间被认为是一个合理的选择。

print(model.generation_config)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 889.46it/s]


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": false,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": false,
  "eos_token_id": [
    151645,
    151643
  ],
  "max_new_tokens": 2048,
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



In [11]:
#定义SFT训练参数
from dataclasses import dataclass
@dataclass
class SFTConfig:
    max_length:int = 1000 #2500 #输入文本的最大长度，超过这个长度的文本将被截断。
    batch_size:int = 1 #每次训练的样本数量，较大的batch_size可以加速训练，但也会增加显存的使用。
    gradient_accumulation_steps:int = 8 #梯度累积的步数，表示在进行一次参数更新之前，累积多少个batch的梯度。
    log_iter:int = 80 #日志记录的迭代次数，表示每隔多少次迭代记录一次训练日志。
    max_lr:float = 1e-4 #训练的最大学习率，较大的学习率可以加速训练，但也可能导致训练不稳定。
    min_lr:float = 1e-5 #训练的最小学习率，较小的学习率可以使训练更稳定，但也可能导致训练过慢。
    warmup_steps:int = 150#学习率预热的步数，表示在训练开始时，学习率从0逐渐增加到最大学习率的步数。

In [12]:
import datasets
from pathlib import Path

data_dir = Path("data/ultrachat")
if not data_dir.exists():
    data_dir = Path("DPO/data/ultrachat")

ultrachat_200k_data = datasets.load_dataset(
    "parquet",
    data_files={"train_sft": str(data_dir / "train_sft-*.parquet")},
)
def tokenize_and_format(data):
    input_ids = tokenizer.apply_chat_template(
        data,
        tokenize = True,
        add_generation_prompt = False,
        truncation = True,
        max_length = SFTConfig.max_length,
    )

    return input_ids["input_ids"]

## 生成训练数据的tokenid
chosen_input_ids_list = []
i = 0
while True:
    data = ultrachat_200k_data['train_sft'][i]['messages']
    data.insert(
        0,
        {"content": "You are a helpful assistant", "role": "system"}
    )
    input_ids = tokenize_and_format(data)
    chosen_input_ids_list.append(input_ids)
    i += 1
    if i % 1000 == 0:
        print(f"已处理{i}条数据")
    if i == 300:#len(ultrachat_200k_data['train_sft']):
        break
print('-' * 70)
print(chosen_input_ids_list[0])
print(tokenizer.decode(chosen_input_ids_list[0]))
print(len(chosen_input_ids_list))
print("数据处理完成！")


----------------------------------------------------------------------
[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 151645, 198, 151644, 872, 198, 9485, 11221, 3796, 311, 3772, 5980, 21386, 320, 78795, 220, 21, 13, 15, 44662, 10392, 2210, 220, 19, 13, 15, 44662, 4270, 44168, 220, 18, 13, 15, 10, 47175, 220, 17, 13, 15, 44662, 34906, 24078, 220, 20, 13, 15, 10, 568, 3555, 6912, 2319, 1079, 358, 1667, 5267, 1925, 697, 25326, 6816, 609, 50219, 25326, 14158, 11, 498, 646, 6707, 1473, 279, 14246, 2168, 315, 264, 1985, 389, 19548, 553, 27362, 825, 315, 279, 6912, 594, 5798, 3419, 5003, 4894, 7771, 11101, 6816, 609, 50219, 25326, 14158, 686, 1431, 3037, 279, 14246, 1985, 2168, 1101, 553, 68607, 916, 429, 1985, 2168, 28574, 624, 21468, 419, 4565, 3796, 311, 678, 14158, 315, 279, 6912, 476, 1101, 3151, 6174, 438, 10007, 304, 279, 1467, 3684, 30, 151645, 198, 151644, 77091, 198, 1986, 4565, 1172, 16790, 311, 11101, 6816, 323, 50219, 25326, 14158, 315, 279, 3772, 5980, 21386, 10007, 304, 279

In [13]:
#训练参数
batch_size = SFTConfig.batch_size
gradient_accumulation_steps = SFTConfig.gradient_accumulation_steps
log_iter = SFTConfig.log_iter
max_lr = SFTConfig.max_lr
min_lr = SFTConfig.min_lr
warmup_steps = SFTConfig.warmup_steps
total_steps = len(chosen_input_ids_list) // batch_size  #计算总的训练步数
optimizer = torch.optim.AdamW(filter(
    lambda p: p.requires_grad,
    model.parameters()
), lr=max_lr) #优化器，AdamW
trainable_parameters_num = sum(p.numel() for p in filter(
    lambda p: p.requires_grad,
    model.parameters())
)  ##全参微调

print(type(chosen_input_ids_list[0]))
print(type(chosen_input_ids_list[0][0]) if len(chosen_input_ids_list[0]) else None)


<class 'list'>
<class 'int'>


In [14]:
##配置logging
import time

with open(f"./Qwen2.5-0.5B-SFT_log.txt", "a") as my_file:
    my_file.write(f' \
        time:{time.strftime("%Y-%m-%d, %H:%M:%S")}, \
        batch_size:{batch_size}, \
        trainable_parameters_num:{trainable_parameters_num}, \
        warmup_steps:{warmup_steps}, \
        max_lr:{max_lr}, \
        min_lr:{min_lr}\n')

#定义一个日志记录函数
def log_call(iters, iters_average_loss):
    with open(f"./Qwen2.5-0.5B-SFT_log.txt", "a") as my_file:
        my_file.write(f' \
            time:{time.strftime("%Y-%m-%d, %H:%M:%S")}, \
            iters:{iters+1}, \
            iters_average_Loss:{iters_average_loss:.4f}\n')



In [15]:
#余弦函数学习率调整
def linear_warmup(current_step, warmup_steps, max_lr):
    if current_step < warmup_steps:
        return max_lr * current_step / warmup_steps
    else:
        return max_lr

def cosine_decay(current_step, warmup_steps, total_steps, max_lr, min_lr):
    if current_step < warmup_steps:
        return linear_warmup(current_step, warmup_steps, max_lr)
    else:
        progress = (current_step - warmup_steps) / (total_steps - warmup_steps) #计算当前训练进度，范围在0到1之间
        decay = 0.5 * (1 + np.cos(np.pi * progress)) #计算余弦衰减因子，范围在0到1之间
        return (max_lr - min_lr) * decay + min_lr

1. $
\text{lr} =
\begin{cases}
\text{max\_lr} \times \dfrac{\text{current\_step}}{\text{warmup\_steps}}, & \text{if } \text{current\_step} < \text{warmup\_steps} \\[1em]
\text{max\_lr}, & \text{otherwise}
\end{cases}
$

参数含义
- **`current_step`**：当前已完成的训练步数（迭代次数）。
- **`warmup_steps`**：预热阶段的总步数。
- **`max_lr`**：预热结束后达到的最大学习率。
具体作用
- 在预热阶段（`current_step < warmup_steps`），学习率从 0 线性增长到 `max_lr`。
- 预热结束后，学习率保持为 `max_lr`。

2. $
\text{progress} = \frac{\text{current\_step} - \text{warmup\_steps}}{\text{total\_steps} - \text{warmup\_steps}} \in [0,1]$
$\text{decay} = \frac{1}{2} \left[ 1 + \cos\left(\pi \cdot \text{progress}\right) \right]  $
$\text{lr} = (\text{max\_lr} - \text{min\_lr}) \cdot \text{decay} + \text{min\_lr}$

参数含义
- **`total_steps`**：总训练步数（包括预热）。
- **`min_lr`**：余弦衰减阶段结束时的最小学习率（最终学习率）。
- 其余参数同线性预热。

具体作用
1. **进度归一化**：`progress` 将预热后的步数映射到 [0,1] 区间，表示剩余训练阶段已完成的比例。
2. **余弦衰减因子**：`decay` 利用了余弦函数在 [0,π] 区间内的性质：
   - 当 `progress = 0` 时，\(\cos(0)=1\)，`decay = 0.5*(1+1)=1`，学习率为 `max_lr`。
   - 当 `progress = 1` 时，\(\cos(\pi)=-1\)，`decay = 0.5*(1-1)=0`，学习率为 `min_lr`。
   - 中间值：余弦函数从 1 单调递减到 -1，`decay` 则从 1 单调递减到 0，下降速度先慢后快再慢（平滑过渡）。
3. **线性插值**：`(max_lr - min_lr) * decay + min_lr` 将衰减因子映射到实际学习率区间。

SFT和预训练，是只将训练的回答部分的loss进行反向传播，并将模型的输入部分（问题部分）的loss进行遮蔽（mask）处理，使其不参与反向传播。这样做的原因是，在SFT和预训练阶段，我们更关注模型生成回答的能力，而不是对输入问题的理解能力。因此，我们只计算回答部分的loss，并将输入部分的loss进行遮蔽处理，以避免对模型的输入部分进行过多的训练，从而更好地提升模型生成回答的质量。
构建损失掩码，对于每个输入样本，我们需要构建一个与输入文本长度相同的掩码（mask），用于指示哪些位置的token应该参与损失计算，哪些位置的token应该被遮蔽掉。

In [16]:
def create_answer_mask(input_ids, tokenizer):
    """
    创建仅对助手回答部分计算损失的掩码

    Args:
        input_ids: 输入token序列 [batch_size, seq_len]
        tokenizer: 分词器

    Returns:
        answer_mask: 助手回答部分为1，其他部分为0的掩码

    对话格式是：
<|im_start|>system\n{system_msg}<|im_end|>\n
<|im_start|>user\n{user_msg}<|im_end|>\n
<|im_start|>assistant\n{assistant_msg}<|im_end|>\n
    """
    batch_size, seq_len = input_ids.shape
    answer_mask = torch.zeros_like(input_ids)

    # 获取每段内容结束标记的token id
    eos_token_id = tokenizer.encode('<|im_end|>')[0]

    for batch_idx in range(batch_size):
        # 找到所有 <|im_end|> 的位置
        eos_positions = torch.where(input_ids[batch_idx] == eos_token_id)[0].tolist()

        if len(eos_positions) < 2:  # 至少需要user和assistant各一个结束标记
            continue

        # 解析对话轮次
        user_ends, assistant_ends = _parse_conversation_turns(eos_positions)

        # 为每个助手回答设置掩码
        _set_answer_masks(
            answer_mask[batch_idx],
            user_ends,
            assistant_ends,
            seq_len
        )

    return answer_mask #返回一个与输入token序列形状相同的掩码，其中助手回答部分为1，其他部分为0


def _parse_conversation_turns(eos_positions):
    """
    解析对话轮次，分离用户和助手的结束位置

    对话格式：
    <|im_start|>system\n{system_msg}<|im_end|>\n
    <|im_start|>user\n{user_msg}<|im_end|>\n
    <|im_start|>assistant\n{assistant_msg}<|im_end|>\n

    eos_positions[0]: system结束 (如果有)
    eos_positions[1]: 第1轮user结束
    eos_positions[2]: 第1轮assistant结束
    eos_positions[3]: 第2轮user结束
    eos_positions[4]: 第2轮assistant结束
    ...
    """
    # 跳过system部分，从第一个user开始
    conversation_eos = eos_positions[1:]  # 去掉system的eos

    # 偶数索引：user结束位置，奇数索引：assistant结束位置
    user_ends = [pos + 1 for pos in conversation_eos[::2]] # 每隔2个取一个，从0开始
    assistant_ends = [pos + 1 for pos in conversation_eos[1::2]] # 每隔2个取一个，从1开始

    return user_ends, assistant_ends #得到用户消息结束位置列表和助手消息结束位置列表


def _set_answer_masks(mask, user_ends, assistant_ends, seq_len):
    """
    为助手回答部分设置掩码

    Args:
        mask: 当前样本的掩码 [seq_len]
        user_ends: 用户消息结束位置列表
        assistant_ends: 助手消息结束位置列表
        seq_len: 序列长度
    """
    num_user_turns = len(user_ends)
    num_assistant_turns = len(assistant_ends)

    if num_user_turns == num_assistant_turns:
        # 完整对话：每轮都有用户问题和助手回答
        for user_end, assistant_end in zip(user_ends, assistant_ends):
            answer_start = user_end + 3  # 跳过 <|im_start|>assistant\n
            answer_end = assistant_end  # 包含 <|im_end|>，让模型学会何时停止
            mask[answer_start:answer_end] = 1

    elif num_user_turns == num_assistant_turns + 1:
        # 对于最后一轮被截断的情况，用户消息有一个额外的结束标记，但没有对应的助手回答结束标记

        # 处理完整的对话轮次
        for user_end, assistant_end in zip(user_ends[:-1], assistant_ends):
            answer_start = user_end + 3
            answer_end = assistant_end
            mask[answer_start:answer_end] = 1

        # 处理最后一轮被截断的助手回答
        last_user_end = user_ends[-1]
        last_answer_start = last_user_end + 3
        mask[last_answer_start:] = 1  # 到序列结尾





In [17]:
#测试掩码函数
test_input_ids = torch.tensor([chosen_input_ids_list[0]])  #使用第一个样本的输入token序列进行测试
answer_mask = create_answer_mask(test_input_ids, tokenizer)
print("输入token序列:", test_input_ids)
print("生成的回答掩码:", answer_mask)
print("输入文本:", tokenizer.decode(test_input_ids[0]))
print("掩码对应的文本:", tokenizer.decode(test_input_ids[0][answer_mask[0] == 1]))
#看一下是否将停止标记也包含在回答掩码中，找<|im_end|>的在输入token序列中的位置和掩码中对应位置的值
eos_token_id = tokenizer.encode('<|im_end|>')[0]
eos_positions = torch.where(test_input_ids[0] == eos_token_id)[0].tolist()
print("输入token序列中<|im_end|>的位置:", eos_positions)
for pos in eos_positions:
    print(f"位置 {pos} 的掩码值: {answer_mask[0, pos].item()}")

输入token序列: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847, 151645,
            198, 151644,    872,    198,   9485,  11221,   3796,    311,   3772,
           5980,  21386,    320,  78795,    220,     21,     13,     15,  44662,
          10392,   2210,    220,     19,     13,     15,  44662,   4270,  44168,
            220,     18,     13,     15,     10,  47175,    220,     17,     13,
             15,  44662,  34906,  24078,    220,     20,     13,     15,     10,
            568,   3555,   6912,   2319,   1079,    358,   1667,   5267,   1925,
            697,  25326,   6816,    609,  50219,  25326,  14158,     11,    498,
            646,   6707,   1473,    279,  14246,   2168,    315,    264,   1985,
            389,  19548,    553,  27362,    825,    315,    279,   6912,    594,
           5798,   3419,   5003,   4894,   7771,  11101,   6816,    609,  50219,
          25326,  14158,    686,   1431,   3037,    279,  14246,   1985,   2168,
           1101, 

In [18]:
model.train()
training_losses = []
model.zero_grad()
skipped_batches_count = 0 #对于回答部分数据不足的批次进行跳过统计

total_batches = len(chosen_input_ids_list) // batch_size
total_update_steps = (total_batches + gradient_accumulation_steps - 1) // gradient_accumulation_steps #计算总的更新步数，考虑到最后一个批次可能不足以进行完整的梯度累积，因此使用了向上取整的方式来计算总的更新步数。

#更新步
update_step = 0
for epoch in range(10): #可以根据需要增加训练轮数
    for batch_idx in range(total_batches):
        ## ==================== 数据准备阶段 ====================

        # 获取当前批次的原始数据
        current_batch_sequences = chosen_input_ids_list[batch_idx * batch_size:(batch_idx + 1) * batch_size]

        # 计算当前批次的最大序列长度，用于padding对齐
        max_sequence_length = max([len(sequence) for sequence in current_batch_sequences])

        ### 对批次数据进行右填充，使所有序列长度一致以便并行计算
        padded_sequences_list = []
        pad_token_id = model.generation_config.eos_token_id[-1] #将EOS token作为填充token进行右填充

        for seq_idx in range(batch_size):
            original_sequence = current_batch_sequences[seq_idx]
            padding_length = max_sequence_length - len(original_sequence) #计算需要填充的长度

            # 使用EOS token进行右填充
            padded_sequence = torch.nn.functional.pad(
                torch.tensor(original_sequence),
                (0, padding_length), #这个表示在左侧不填充，在右侧填充padding_length个token
                mode='constant', #填充模式为常数填充，即使用指定的值进行填充
                value=pad_token_id #填充的值为pad_token_id，即EOS token的id
            ).tolist() #将填充后的序列转换为列表形式

            padded_sequences_list.append(padded_sequence) #将填充后的序列添加到批次列表中

        # 转换为张量
        batch_input_tensor = torch.tensor(padded_sequences_list)

        ## ==================== 构建输入输出对 ====================

        # 构建因果语言模型的输入输出对：x->y（下一个词预测）,输入是前n-1个token，标签是后n-1个token
        model_inputs = batch_input_tensor[:, :-1].to(device)    # 输入：前n-1个token
        target_labels = batch_input_tensor[:, 1:].to(device)    # 标签：后n-1个token

        ## ==================== 构建训练掩码 ====================

        # 构建掩码矩阵来控制损失计算范围
        # 1. padding_mask：标识哪些位置是填充token（不计算损失）
        # 2. answer_mask：标识哪些位置是助手回答部分（只对回答计算损失）

        ### 【填充掩码】：非填充token为1，填充token为0
        padding_mask = torch.where(target_labels == pad_token_id, 0, 1)  #生成填充掩码，target_labels中等于pad_token_id的位置为0（表示填充），其他位置为1（表示非填充）

        ### 【回答掩码】：只有助手回答部分为1，其他部分为0
        assistant_answer_mask = create_answer_mask(model_inputs, tokenizer)

        ### 【组合掩码】：同时满足"非填充"且"是回答部分"的token才计算损失
        final_loss_mask = (assistant_answer_mask & padding_mask)
        '''
        [<|im_start|>, user, 天气, 如何, ?, <|im_end|>, <|im_start|>, assistant, 今天, 晴, <|im_end|>, PAD, PAD]
        assistant_answer_mask[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]
        padding_mask[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
        final_loss_mask[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]
        '''
        ## ==================== 批次有效性检查 ====================

        # 检查当前批次是否有效：如果某个样本的回答部分完全为空，则跳过该批次
        # 这种情况通常发生在问题过长导致回答部分被截断时
        tokens_per_sample = final_loss_mask.sum(dim=-1)  # 每个样本的有效回答token数
        min_answer_tokens = tokens_per_sample.min().item()  # 最少的有效token数

        if min_answer_tokens == 0:
            print(f'⚠️ 跳过第{batch_idx + 1}批次：回答部分数据不足')
            skipped_batches_count += 1
            continue  # 跳过当前批次

        ## ==================== 模型前向传播 ====================

        # 执行前向传播，获取模型预测的logits
        model_logits = model(model_inputs).logits

        ## ==================== 损失计算 ====================

        # 计算带掩码的交叉熵损失
        # 步骤：logits -> softmax -> log -> gather -> 负对数似然 -> 掩码过滤 -> 平均

        # 1. 计算每个token的负对数似然损失
        log_probabilities = torch.log(torch.softmax(model_logits, dim=-1))#[batch_size, seq_len, vocab_size]
        gathered_log_probs = torch.gather(log_probabilities, dim=-1, index=target_labels.unsqueeze(2))
        '''
        gathered_log_probs的形状为[batch_size, seq_len, 1]，其中每个位置的值是模型预测的对应标签token的log概率
        gather根据target_labels中的token id从log_probabilities中提取对应位置的log概率值。通过unsqueeze(2)，我们将target_labels的形状从[batch_size, seq_len]扩展为[batch_size, seq_len, 1]，与log_probabilities的形状匹配，从而正确地提取每个token的log概率。
        '''
        negative_log_likelihood = gathered_log_probs * (-1)  # 负对数似然
        token_losses = negative_log_likelihood.squeeze(2) #将形状从[batch_size, seq_len, 1]变为[batch_size, seq_len]，每个位置的值是对应token的损失值

        # 2. 应用掩码并计算每个样本的平均损失
        masked_token_losses = torch.mul(token_losses, final_loss_mask) #只保留回答部分的损失，其他部分的损失被置为0
        sample_losses = masked_token_losses.sum(dim=-1) / final_loss_mask.sum(dim=-1) #计算每个样本的平均损失，分母是有效token的数量，确保只对回答部分进行平均

        # 3. 计算批次平均损失并应用梯度累积
        batch_average_loss = torch.nanmean(sample_losses) / gradient_accumulation_steps #计算批次平均损失，并除以梯度累积步数进行缩放，以便在累积多个批次后再进行一次权重更新

        ## ==================== 反向传播和优化 ====================

        # 反向传播计算梯度
        batch_average_loss.backward()

        # 梯度累积：只在累积步数达到或最后一个批次时更新权重
        is_accumulation_step = (batch_idx + 1) % gradient_accumulation_steps == 0
        is_final_batch = (batch_idx + 1) == total_batches

        if is_accumulation_step or is_final_batch:
          update_step += 1

          # 动态调整学习率（余弦衰减 + 预热）
          current_learning_rate = cosine_decay(
            update_step,
            warmup_steps,
            total_update_steps,
            max_lr,
            min_lr
          )
          for param_group in optimizer.param_groups:
            param_group["lr"] = current_learning_rate

          optimizer.step()
          optimizer.zero_grad()



        ## ==================== 训练日志记录 ====================

        # 记录当前批次的损失（还原梯度累积的缩放）
        actual_batch_loss = batch_average_loss.item() * gradient_accumulation_steps
        training_losses.append(actual_batch_loss)

        # 定期输出训练进度
        should_log = (batch_idx + 1) % log_iter == 0 or is_final_batch

        if should_log:
            # 计算最近几个批次的平均损失
            recent_losses = training_losses[-log_iter:]
            recent_average_loss = np.nanmean(recent_losses)

            # 输出训练状态
            current_time = time.strftime("%Y-%m-%d %H:%M:%S")
            print(f'⏰ 时间: {current_time} | '
                  f'📊 批次: {batch_idx + 1}/{total_batches} | '
                  f'更新步: {update_step}/{total_update_steps} | '
                  f'📈 最近{len(recent_losses)}批次平均损失: {recent_average_loss:.4f} | '
                  f'🎯 学习率: {current_learning_rate:.2e}')

            # 调用外部日志记录函数
            log_call(batch_idx, recent_average_loss)

## ==================== 训练完成总结 ====================

print("🎉 训练完成!")
print(f'📊 训练统计:')
print(f'   - 总批次数: {total_batches}')
print(f'   - 跳过批次数: {skipped_batches_count}')
print(f'   - 有效批次数: {total_batches - skipped_batches_count}')
print(f'   - 最终平均损失: {np.nanmean(training_losses[-100:]):.4f}')

if skipped_batches_count > 0:
    skip_ratio = skipped_batches_count / total_batches * 100
    print(f'⚠️ 跳过批次占比: {skip_ratio:.2f}%')
    if skip_ratio > 10:
        print('💡 建议: 跳过批次过多，考虑增加最大序列长度或优化数据预处理')

⚠️ 跳过第11批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:20:08 | 📊 批次: 80/300 | 更新步: 10/38 | 📈 最近79批次平均损失: 1.4535 | 🎯 学习率: 6.67e-06
⏰ 时间: 2026-04-02 14:20:27 | 📊 批次: 160/300 | 更新步: 20/38 | 📈 最近80批次平均损失: 1.4230 | 🎯 学习率: 1.33e-05
⚠️ 跳过第187批次：回答部分数据不足
⚠️ 跳过第207批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:20:45 | 📊 批次: 240/300 | 更新步: 30/38 | 📈 最近80批次平均损失: 1.5374 | 🎯 学习率: 2.00e-05
⚠️ 跳过第262批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:20:59 | 📊 批次: 300/300 | 更新步: 38/38 | 📈 最近80批次平均损失: 1.4813 | 🎯 学习率: 2.53e-05
⚠️ 跳过第11批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:21:18 | 📊 批次: 80/300 | 更新步: 48/38 | 📈 最近80批次平均损失: 1.4249 | 🎯 学习率: 3.20e-05
⏰ 时间: 2026-04-02 14:21:36 | 📊 批次: 160/300 | 更新步: 58/38 | 📈 最近80批次平均损失: 1.3843 | 🎯 学习率: 3.87e-05
⚠️ 跳过第187批次：回答部分数据不足
⚠️ 跳过第207批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:21:55 | 📊 批次: 240/300 | 更新步: 68/38 | 📈 最近80批次平均损失: 1.4915 | 🎯 学习率: 4.53e-05
⚠️ 跳过第262批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:22:09 | 📊 批次: 300/300 | 更新步: 76/38 | 📈 最近80批次平均损失: 1.4329 | 🎯 学习率: 5.07e-05
⚠️ 跳过第11批次：回答部分数据不足
⏰ 时间: 2026-04-02 14:22:28 | 📊 批次: 80/300 | 更新步: 

因为显卡显存问题，采用量化和QLora微调相结合的方式进行SFT训练。

In [19]:
model.save_pretrained('models/Qwen2.5-0.5B-SFT-falsesample-r16')
tokenizer.save_pretrained('models/Qwen2.5-0.5B-SFT-falsesample-r16')

merge_save_path = Path("models/Qwen2.5-0.5B-SFT-merged-falsesample")

merged_model = model.merge_and_unload()

merge_save_path.mkdir(parents=True, exist_ok=True)

# 保存前修正 generation_config（贪心模式）
merged_model.generation_config.do_sample = False
merged_model.generation_config.temperature = None
merged_model.generation_config.top_p = None
merged_model.generation_config.top_k = None

merged_model.save_pretrained(
    str(merge_save_path),
    safe_serialization=True
)
tokenizer.save_pretrained(str(merge_save_path))


print(f"合并后的模型已保存到: {merge_save_path}")



D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\peft\tuners\lora\bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]

合并后的模型已保存到: models\Qwen2.5-0.5B-SFT-merged-falsesample
